# Few-Shot Example Extractor + System Prompt Builder
## Area Classifier — ZHC

**Obiettivo**: estrarre automaticamente i migliori esempi dal dataset per costruire
un few-shot prompt GPT che batta il LinearSVC sulle classi problematiche.

### Strategia di selezione esempi
Usiamo il **`decision_function` del LinearSVC già addestrato**: gli esempi con il
margine più alto dalla frontiera di decisione sono i più *prototipici* della classe.
Questo è fondamentale: non vogliamo esempi ambigui, vogliamo quelli che insegnano
a GPT esattamente il vocabolario e il pattern di ogni area.

| Criterio | Perché |
|---|---|
| Alto margine decision_function | Massima certezza del modello = esempio prototypico |
| Solo dal train set | Nessun data leakage dal test set |
| Lunghezza 20-120 parole | Abbastanza contesto, non troppo token |
| Diversità tematica | 3 esempi per classe con contenuto diverso |
| Focus sulle classi difficili | Più esempi per sistema381, area_territoriale, area_sistemistica |

## STEP 1 — Import e caricamento

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('..').resolve()
CSV_PATH      = BASE_DIR / 'data'    / 'dataset_clean.csv'
MODEL_DIR     = BASE_DIR / 'modelli' / 'area_v2'
OUTPUT_DIR    = BASE_DIR / 'data'

df = pd.read_csv(CSV_PATH, parse_dates=['data_creazione'])
print(f'Dataset: {len(df):,} righe, {len(df.columns)} colonne')
print(f'Colonne keyword: {sum(1 for c in df.columns if c.startswith("kw_"))}')

Dataset: 58,385 righe, 29 colonne
Colonne keyword: 18


## STEP 2 — Preparazione dataset (replica esatta AreaClassifier)

In [2]:
# ── Stessa pulizia usata in AreaClassifier_VSCode.ipynb ──────────────────────
df['area_v2'] = df['area'].replace({'Hardware': 'area_sistemistica'})

CLASSI_DROP = ['business_intelligence', 'protocollo_delibere']
df.loc[df['area_v2'].isin(CLASSI_DROP), 'area_v2'] = np.nan

conteggi = df['area_v2'].value_counts()
classi_rumore = conteggi[conteggi < 10].index.tolist()
df.loc[df['area_v2'].isin(classi_rumore), 'area_v2'] = np.nan

df_validi = df.dropna(subset=['area_v2', 'testo_input']).copy()

# ── Split temporale identico al training ─────────────────────────────────────
SOGLIA_SPLIT = '2025-11-01'
mask_train = df_validi['data_creazione'] < SOGLIA_SPLIT

df_train = df_validi[mask_train].copy()
df_test  = df_validi[~mask_train].copy()

print(f'Train: {len(df_train):,}   Test: {len(df_test):,}')
print(f'\nClassi nel train:')
print(df_train['area_v2'].value_counts().to_string())

Train: 41,828   Test: 10,321

Classi nel train:
area_v2
area_personale                     11087
ciclo_passivo                       8927
ciclo_attivo                        7359
area_sanitaria                      7323
rendicontazione_flussi              2459
protocollo_documentale_delibere     1896
area_sistemistica                   1500
sistema381                           834
area_territoriale                    443


## STEP 3 — Caricamento modello LinearSVC e calcolo decision scores

In [3]:
# ── STEP 3 corretto ───────────────────────────────────────────────────────────
import joblib, json
import numpy as np

MODEL_DIR  = BASE_DIR / 'modelli' / 'area_v2'
EMB_DIR    = BASE_DIR / 'embeddings'

# Il "pipeline" è direttamente un LinearSVC
svc = joblib.load(MODEL_DIR / 'classificatore_svc.pkl')   # rinomina se il file si chiama diversamente
print(type(svc))   # deve stampare <class 'sklearn.svm._classes.LinearSVC'>

with open(MODEL_DIR / 'metadata.json') as f:
    meta = json.load(f)

CLASSI   = meta['classi']
KW_COLS  = meta['keyword_cols']
print('Classi:', CLASSI)
print('Keyword cols:', KW_COLS)

<class 'sklearn.svm._classes.LinearSVC'>
Classi: ['area_personale', 'area_sanitaria', 'area_sistemistica', 'area_territoriale', 'ciclo_attivo', 'ciclo_passivo', 'protocollo_documentale_delibere', 'rendicontazione_flussi', 'sistema381']
Keyword cols: ['kw_s381_codice', 'kw_s381_rapportino', 'kw_s381_timbrate', 'kw_s381_calendario_presenze', 'kw_ter_unodomo', 'kw_ter_distretto', 'kw_pas_iva', 'kw_pas_fornitore', 'kw_pas_cespiti', 'kw_pas_prima_nota', 'kw_pas_ammortamento', 'kw_pas_analitica', 'kw_pas_reverse', 'kw_att_retta', 'kw_att_pagopa', 'kw_att_sdd', 'kw_att_portale_utenti', 'kw_att_fattura_elettronica']


## STEP 4 — Estrazione esempi prototipici per classe

In [4]:
# ── STEP 4 corretto: ricostruisci X dal train usando .npy già calcolati ───────
import scipy.sparse as sp
from sklearn.preprocessing import OneHotEncoder

# 1. Carica embeddings train già calcolati su Colab (stessa logica di AreaClassifier_VSCode)
emb_train = np.load(EMB_DIR / 'area_v3_e5_train.npy')   # shape (n_train, 768)
print(f'Embeddings train: {emb_train.shape}')

# 2. Ricostruisci OHE priorità iniziale cliente
#    (stesso encoder usato in training — se hai salvato l'encoder caricalo,
#     altrimenti rifallo con le stesse categorie)
try:
    ohe = joblib.load(MODEL_DIR / 'ohe_priorita.pkl')
    ohe_train = ohe.transform(df_train[['priorita_iniziale_cliente']].fillna('P3'))
except FileNotFoundError:
    # Rifai l'encoder con le stesse categorie del training
    ohe = OneHotEncoder(categories=[['P1','P2','P3','P4']], sparse_output=True, handle_unknown='ignore')
    ohe.fit(df_train[['priorita_iniziale_cliente']].fillna('P3'))
    ohe_train = ohe.transform(df_train[['priorita_iniziale_cliente']].fillna('P3'))

print(f'OHE shape: {ohe_train.shape}')

# 3. Keyword booleane — già in df_train come colonne kw_*
kw_train = sp.csr_matrix(df_train[KW_COLS].values.astype(float))
print(f'Keyword shape: {kw_train.shape}')

# 4. Concatena esattamente come in training
X_train = sp.hstack([
    sp.csr_matrix(emb_train),
    ohe_train,
    kw_train
])
print(f'X_train finale: {X_train.shape}')

# 5. Calcola decision_function
scores = svc.decision_function(X_train)
scores_df = pd.DataFrame(scores, columns=svc.classes_, index=df_train.index)
print(f'Decision scores: {scores_df.shape}  — tutto ok, procedi con Step 4b')

Embeddings train: (41828, 768)
OHE shape: (41828, 4)
Keyword shape: (41828, 18)
X_train finale: (41828, 790)
Decision scores: (41828, 9)  — tutto ok, procedi con Step 4b


In [5]:
# ── Parametri di selezione esempi ────────────────────────────────────────────

# Lunghezza testo accettabile (in parole) — troppo corto = poco contesto,
# troppo lungo = troppi token nel prompt
MIN_PAROLE = 15
MAX_PAROLE = 150

# Numero di esempi per classe
# Le classi difficili (F1 basso nel LinearSVC) ricevono più esempi
N_ESEMPI_BASE      = 2
N_ESEMPI_DIFFICILI = 3

CLASSI_DIFFICILI = {
    'area_sistemistica',    # F1 0.51 — usata come catch-all da GPT
    'area_territoriale',    # F1 0.39 — classe piccola, pochi esempi nel train
    'sistema381',           # F1 0.50 — vocabolario molto specifico
    'ciclo_attivo',         # F1 0.70 — overlap con ciclo_passivo
    'ciclo_passivo',        # F1 0.74 — overlap con ciclo_attivo
}

print(f'MIN_PAROLE: {MIN_PAROLE}  |  MAX_PAROLE: {MAX_PAROLE}')
print(f'Esempi per classe normale:   {N_ESEMPI_BASE}')
print(f'Esempi per classe difficile: {N_ESEMPI_DIFFICILI}')
print(f'Classi difficili: {CLASSI_DIFFICILI}')

MIN_PAROLE: 15  |  MAX_PAROLE: 150
Esempi per classe normale:   2
Esempi per classe difficile: 3
Classi difficili: {'ciclo_attivo', 'area_sistemistica', 'sistema381', 'ciclo_passivo', 'area_territoriale'}


In [6]:
# ── Selezione esempi per ogni classe ─────────────────────────────────────────

def estrai_esempi(classe, n_esempi):
    """
    Estrae N esempi prototipici per una classe:
    1. Filtra solo i ticket di quella classe nel train set
    2. Applica filtro di lunghezza
    3. Ordina per decision_function score (alto = più prototipico)
    4. Diversifica: seleziona esempi distanti tra loro nel ranking
    """
    mask_classe = df_train['area_v2'] == classe
    idx_classe  = df_train[mask_classe].index
    
    # Filtra per lunghezza
    n_parole = df_train.loc[idx_classe, 'testo_input'].str.split().str.len()
    mask_len = (n_parole >= MIN_PAROLE) & (n_parole <= MAX_PAROLE)
    idx_filtrati = idx_classe[mask_len]
    
    if len(idx_filtrati) == 0:
        print(f'  WARN: nessun esempio nella lunghezza target per {classe}')
        idx_filtrati = idx_classe  # fallback senza filtro lunghezza
    
    # Score per questa classe
    score_classe = scores_df.loc[idx_filtrati, classe].sort_values(ascending=False)
    
    # Diversifica: prendi il top, poi scorri il ranking a step
    # es. con n=3 prendiamo indice 0, len/3, 2*len/3
    n_disponibili = len(score_classe)
    if n_disponibili <= n_esempi:
        idx_scelti = score_classe.index.tolist()
    else:
        step = n_disponibili // (n_esempi + 1)
        posizioni = [0] + [min(step * (i+1), n_disponibili-1) for i in range(n_esempi - 1)]
        idx_scelti = [score_classe.index[p] for p in posizioni]
    
    esempi = []
    for idx in idx_scelti:
        row = df_train.loc[idx]
        esempi.append({
            'classe':   classe,
            'score':    round(float(scores_df.loc[idx, classe]), 3),
            'n_parole': int(row['testo_input'].split().__len__()),
            'oggetto':  str(row.get('oggetto', row['testo_input'].split('\n')[0][:80])),
            'testo':    str(row['testo_input'])[:600],  # tronca a 600 char
        })
    
    return esempi


# Estrai per tutte le classi
tutti_esempi = {}

for classe in CLASSI:
    n = N_ESEMPI_DIFFICILI if classe in CLASSI_DIFFICILI else N_ESEMPI_BASE
    esempi = estrai_esempi(classe, n)
    tutti_esempi[classe] = esempi
    print(f'{classe:<40} → {len(esempi)} esempi  (score range: {esempi[0]["score"]:.2f} … {esempi[-1]["score"]:.2f})')

print(f'\nTotale esempi: {sum(len(v) for v in tutti_esempi.values())}')

area_personale                           → 2 esempi  (score range: 3.08 … 0.90)
area_sanitaria                           → 2 esempi  (score range: 2.64 … 0.57)
area_sistemistica                        → 3 esempi  (score range: 2.52 … 0.41)
area_territoriale                        → 3 esempi  (score range: 5.46 … 0.77)
ciclo_attivo                             → 3 esempi  (score range: 2.51 … -0.05)
ciclo_passivo                            → 3 esempi  (score range: 4.79 … 0.10)
protocollo_documentale_delibere          → 2 esempi  (score range: 4.74 … 1.47)
rendicontazione_flussi                   → 2 esempi  (score range: 3.36 … 0.99)
sistema381                               → 3 esempi  (score range: 8.13 … 0.95)

Totale esempi: 23


## STEP 5 — Ispezione manuale degli esempi

In [7]:
# ── Stampa gli esempi per ispezione manuale ───────────────────────────────────
# IMPORTANTE: scorri questi risultati e verifica che gli esempi siano
# effettivamente rappresentativi. Se un esempio sembra sbagliato o ambiguo,
# rimuovilo dalla lista manualmente nello step successivo.

for classe, esempi in tutti_esempi.items():
    print(f'\n{"="*70}')
    print(f'  {classe.upper()}  ({len(esempi)} esempi)')
    print(f'{"="*70}')
    for i, ex in enumerate(esempi, 1):
        print(f'\n  [{i}] Score: {ex["score"]:+.2f}  | Parole: {ex["n_parole"]}')
        print(f'  Oggetto: {ex["oggetto"]}')
        testo_preview = ex['testo'][:300].replace('\n', ' ')
        print(f'  Testo:   {testo_preview}...' if len(ex['testo']) > 300 else f'  Testo:   {testo_preview}')


  AREA_PERSONALE  (2 esempi)

  [1] Score: +3.08  | Parole: 62
  Oggetto: Cedolini stipendi e tabulato timbrature non presente sul portale personale ultim
  Testo:   Cedolini stipendi e tabulato timbrature non presente sul portale personale ultima dipendente assunta Orario reperibilità contatto: 08.15 - 15.45 TUTTI I GIORNITelefono: 0471/978752  Buongiorno, la collaboratrice sig.ra Cusimano Nilde, assunta presso questa Fondazione dal 01/10/2024, non riceve a tut...

  [2] Score: +0.90  | Parole: 43
  Oggetto: ZAINAGHI EMMA Orario reperibilità contatto: Telefono: 3923417819
  Testo:   ZAINAGHI EMMA Orario reperibilità contatto: Telefono: 3923417819  Buongiorno, da un controllo eseguito sul portale mancano le timbrature della dipendente ZAINAGHI EMMA da Gennaio 2025. Chiedo di essere contatta da lunedì 10 marzo, in quanto domani sarò assente dal lavoro.   Cordiali saluti.   Nicoli...

  AREA_SANITARIA  (2 esempi)

  [1] Score: +2.64  | Parole: 33
  Oggetto: Blocco CSS all'apertura del d

## STEP 6 — Costruzione System Prompt ottimizzato

Il prompt segue questa struttura:
1. **Ruolo e contesto aziendale** — chi sei e cosa fai
2. **Classi con descrizione dettagliata ZHC** — vocabolario specifico per ogni area  
3. **Regole di disambiguazione** — le coppie problematiche con criteri espliciti
4. **Esempi few-shot per classe** — i prototipi estratti sopra
5. **Formato output** — Pydantic JSON

In [ ]:
# ── Descrizioni ZHC per ogni classe ──────────────────────────────────────────
# Costruite combinando:
# - Nomi moduli SW dal dataset (crm_modulo_sw_c)
# - Keyword discriminanti dall'analisi chi-quadro
# - Conoscenza del dominio ZHC (healthcare software)

DESCRIZIONI_CLASSI = {
    "area_personale": (
        "Gestione risorse umane, paghe e presenze del personale. "
        "Include: stipendi, buste paga, cedolini, presenze, timbrature, "
        "ferie, permessi, assunzioni, contratti, rilevazione presenze, "
        "gestione turni, HR. Moduli tipici: SO-Pers-Stipendi, "
        "PER-Rilevazione Presenze, Gestione Stipendi, HR Presenze Project."
    ),
    "area_sanitaria": (
        "Gestione clinica e sanitaria di pazienti e ospiti in strutture "
        "sociosanitarie. Include: cartelle cliniche, schede pazienti, "
        "pazienti ADI (assistenza domiciliare integrata), RSA, case di riposo, "
        "farmacia interna, ECO, gestione ospiti, accettazione, dimissioni, "
        "prestazioni sanitarie, infermieri, medici, terapie. "
        "Moduli tipici: ECO-Farmacia, Ospiti Web, ADI."
    ),
    "area_sistemistica": (
        "Problemi ESCLUSIVAMENTE tecnici di infrastruttura, NON funzionalità applicative. "
        "Usa questa classe SOLO SE il ticket riguarda: "
        "installazione/aggiornamento versione del software, "
        "problemi di rete o connettività, accessi/permessi utente al sistema, "
        "certificati SSL, backup, configurazione server, hardware fisico (PC, stampanti, tablet). "
        "NON usare per: errori di calcolo, dati mancanti, funzionalità che non funzionano correttamente — "
        "quelli vanno nell'area funzionale di competenza (personale, sanitaria, ecc.)."
    ),
    "area_territoriale": (
        "Servizi domiciliari e territoriali per enti pubblici. "
        "Include: software UnoDomo (piattaforma ADI Lombardia), "
        "distretto sanitario, servizi territoriali, piattaforme regionali "
        "per assistenza domiciliare, rendicontazione ADI verso ASL/ATS, "
        "piani di assistenza domiciliare. KEYWORD DISTINTIVA: 'unodomo', 'distretto', "
        "'ADI Lombardia', 'piattaforma ADI'."
    ),
    "ciclo_attivo": (
        "Fatturazione attiva, incassi e gestione crediti verso clienti/ospiti. "
        "Include: rette ospiti, fatture a clienti, PagoPA, SDD (addebito diretto), "
        "portale utenti, fattura elettronica B2C, CRA (Case Riposo e Anziani), "
        "OspitiWeb, incassi, estratti conto clienti, note credito attive. "
        "ATTENZIONE: se parla di RICEVERE fatture da fornitori → ciclo_passivo."
    ),
    "ciclo_passivo": (
        "Contabilità fornitori, acquisti e pagamenti verso terzi. "
        "Include: fatture passive (ricevute da fornitori), IVA, cespiti, "
        "ammortamenti, prima nota, contabilità analitica, reverse charge, "
        "registrazione fatture fornitori, scadenzario passivo, pagamenti fornitori, "
        "Contabilità Economica. Moduli tipici: AMM-Economica, AMM-Cespiti. "
        "ATTENZIONE: se parla di EMETTERE fatture a clienti/ospiti → ciclo_attivo."
    ),
    "protocollo_documentale_delibere": (
        "Gestione documentale, protocollo informatico e delibere. "
        "Include: protocollo in entrata/uscita, archiviazione documenti, "
        "delibere del CDA, determine, atti amministrativi, firma digitale, "
        "gestione pratiche, workflow documentale, albo pretorio, PEC istituzionale."
    ),
    "rendicontazione_flussi": (
        "Rendicontazione verso enti pubblici e gestione flussi regionali. "
        "Include: flussi verso Regione, reportistica obbligatoria verso ASL/ATS, "
        "rendicontazione finanziaria pubblica, SIAF, flussi informativi sanitari, "
        "debito informativo regionale, export dati verso sistemi regionali."
    ),
    "sistema381": (
        "Software gestionale per cooperative sociali (Legge 381/1991). "
        "Include: gestione rapportini di lavoro, timbrate cooperative, "
        "calendario presenze lavoratori soci, SIAF cooperativo, "
        "codici attività L.381, rendicontazione cooperativa sociale. "
        "KEYWORD DISTINTIVA: '381', 'rapportino', 'rapportini', 'timbrate', "
        "'cooperativa sociale', 'calendario presenze' (in contesto cooperativo)."
    ),
}

print('Descrizioni classi definite:', len(DESCRIZIONI_CLASSI))

Descrizioni classi definite: 9


In [ ]:
def build_system_prompt(tutti_esempi: dict, descrizioni: dict) -> str:
    """
    Costruisce il system prompt few-shot ottimizzato.
    
    Struttura:
    1. Ruolo e compito
    2. Classi con descrizione
    3. Regole di disambiguazione esplicite (coppie problematiche)
    4. Esempi few-shot per ogni classe
    5. Istruzioni output
    """
    
    # ── 1. Ruolo ──────────────────────────────────────────────────────────────
    prompt = """Sei un classificatore specializzato di ticket di supporto per Zucchetti Healthcare (ZHC).
ZHC produce software gestionale per strutture sociosanitarie italiane (RSA, cooperative sociali, ASL).

COMPITO: Dato oggetto e descrizione di un ticket, identifica l'AREA di competenza.
"""

    # ── 2. Classi con descrizione ─────────────────────────────────────────────
    prompt += "\n## AREE DISPONIBILI\n"
    for classe, desc in descrizioni.items():
        prompt += f"\n**{classe}**\n{desc}\n"

    # ── 3. Regole di disambiguazione ──────────────────────────────────────────
    prompt += """
## REGOLE DI DISAMBIGUAZIONE

Le seguenti coppie generano confusione frequente. Applica questi criteri:

**ciclo_attivo vs ciclo_passivo**
- "fattura" o "IVA" da soli NON bastano a distinguere
- ciclo_ATTIVO: rette, PagoPA, SDD, OspitiWeb, incassi DA ospiti/clienti
- ciclo_PASSIVO: fornitori, cespiti, ammortamenti, acquisti, fatture RICEVUTE

**area_personale vs ciclo_passivo**
- area_personale: stipendi, buste paga, presenze, HR → il DIPENDENTE è il soggetto
- ciclo_passivo: contabilità fornitori → il FORNITORE è il soggetto

**area_sanitaria vs area_personale**
- area_sanitaria: il soggetto è un PAZIENTE/OSPITE della struttura
- area_personale: il soggetto è un DIPENDENTE della struttura

**area_sistemistica — regola RESTRITTIVA**
Usa area_sistemistica SOLO se il ticket contiene almeno una di queste situazioni:
- "installazione", "aggiornamento versione", "update", "deploy"
- "non riesco ad accedere", "utente bloccato", "reset password"  
- "server", "database giù", "connessione", "VPN", "certificato"
- "stampante", "tablet", "PC", problema hardware fisico
SE il ticket dice "il modulo X non calcola Y" oppure "errore nel salvataggio di Z"
→ NON è area_sistemistica, è l'area funzionale di X/Z.

**area_territoriale**
- Usa questa classe SE e SOLO SE compare: 'unodomo', 'uno domo', 'distretto', 'ADI Lombardia', 'piattaforma territoriale'
- Ticket ADI generici (senza questi termini) → area_sanitaria

**sistema381**
- Usa questa classe SE e SOLO SE compare: '381', 'rapportino', 'cooperativa sociale', 'timbrate' (in contesto L.381)
- Presenze generiche senza riferimento 381 → area_personale
"""

    # ── 4. Esempi few-shot ────────────────────────────────────────────────────
    prompt += "\n## ESEMPI PER CLASSE\n"
    prompt += "Di seguito esempi reali di ticket classificati correttamente.\n"

    for classe, esempi in tutti_esempi.items():
        prompt += f"\n### {classe}\n"
        for i, ex in enumerate(esempi, 1):
            # Tronca il testo a 400 char per risparmiare token
            testo_short = ex['testo'][:400]
            if len(ex['testo']) > 400:
                testo_short += '...'
            prompt += f"\nEsempio {i}:\n"
            prompt += f"OGGETTO: {ex['oggetto'][:100]}\n"
            prompt += f"TESTO: {testo_short}\n"
            prompt += f"→ AREA: {classe}\n"

    # ── 5. Istruzioni output ──────────────────────────────────────────────────
    prompt += """
## OUTPUT

Rispondi ESCLUSIVAMENTE con un oggetto JSON con questi campi:
- area: una delle classi elencate sopra (stringa esatta)
- confidenza: float 0.0-1.0 (usa valori bassi < 0.6 se sei incerto tra due classi)
- reasoning: massimo una frase breve che spiega il segnale chiave

Nessun testo aggiuntivo fuori dal JSON.
"""

    return prompt


SYSTEM_PROMPT = build_system_prompt(tutti_esempi, DESCRIZIONI_CLASSI)

n_token_stimati = len(SYSTEM_PROMPT.split()) * 1.3
print(f'System prompt generato:')
print(f'  Caratteri: {len(SYSTEM_PROMPT):,}')
print(f'  Parole:    {len(SYSTEM_PROMPT.split()):,}')
print(f'  Token stimati: ~{n_token_stimati:.0f}')
print(f'  Costo stimato per ticket: ~${(n_token_stimati / 1_000_000) * 0.15:.5f} input')

System prompt generato:
  Caratteri: 15,561
  Parole:    2,053
  Token stimati: ~2669
  Costo stimato per ticket: ~$0.00040 input


## STEP 7 — Salvataggio prompt e esempi

In [10]:
# ── Salva il prompt come file di testo ────────────────────────────────────────
prompt_path = OUTPUT_DIR / 'gpt_system_prompt_fewshot.txt'
with open(prompt_path, 'w', encoding='utf-8') as f:
    f.write(SYSTEM_PROMPT)
print(f'System prompt salvato: {prompt_path}')

# ── Salva gli esempi come JSON per ispezione/modifica manuale ─────────────────
esempi_path = OUTPUT_DIR / 'gpt_fewshot_examples.json'
with open(esempi_path, 'w', encoding='utf-8') as f:
    json.dump(tutti_esempi, f, ensure_ascii=False, indent=2)
print(f'Esempi salvati: {esempi_path}')

# ── Preview delle prime 2000 char del prompt ──────────────────────────────────
print('\n' + '='*60)
print('PREVIEW SYSTEM PROMPT (prime 2000 char):')
print('='*60)
print(SYSTEM_PROMPT[:2000])
print('...[continua]...')

System prompt salvato: C:\Users\matteo.segatto\Desktop\TicketClassifier\data\gpt_system_prompt_fewshot.txt
Esempi salvati: C:\Users\matteo.segatto\Desktop\TicketClassifier\data\gpt_fewshot_examples.json

PREVIEW SYSTEM PROMPT (prime 2000 char):
Sei un classificatore specializzato di ticket di supporto per Zucchetti Healthcare (ZHC).
ZHC produce software gestionale per strutture sociosanitarie italiane (RSA, cooperative sociali, ASL).

COMPITO: Dato oggetto e descrizione di un ticket, identifica l'AREA di competenza.

## AREE DISPONIBILI

**area_personale**
Gestione risorse umane, paghe e presenze del personale. Include: stipendi, buste paga, cedolini, presenze, timbrature, ferie, permessi, assunzioni, contratti, rilevazione presenze, gestione turni, HR. Moduli tipici: SO-Pers-Stipendi, PER-Rilevazione Presenze, Gestione Stipendi, HR Presenze Project.

**area_sanitaria**
Gestione clinica e sanitaria di pazienti e ospiti in strutture sociosanitarie. Include: cartelle cliniche, schede paz

## STEP 8 — Classificatore few-shot GPT

In [11]:
import os
import re
import time
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal
from dotenv import load_dotenv

load_dotenv(BASE_DIR / '.env')
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


# ── Modello Pydantic per output strutturato ───────────────────────────────────
AreaType = Literal[
    'area_personale', 'area_sanitaria', 'area_sistemistica', 'area_territoriale',
    'ciclo_attivo', 'ciclo_passivo', 'protocollo_documentale_delibere',
    'rendicontazione_flussi', 'sistema381'
]

class ClassificazioneArea(BaseModel):
    area: AreaType
    confidenza: float
    reasoning: str


def preprocess_testo(oggetto: str, descrizione: str, max_parole: int = 400) -> str:
    """Pulisce e tronca il testo prima di inviarlo a GPT."""
    import re
    testo = f"{oggetto or ''}\n{descrizione or ''}"
    testo = re.sub(r'<[^>]+>', ' ', testo)                        # rimuovi HTML
    testo = re.sub(r'Orario reperibilità.*', '', testo, flags=re.DOTALL | re.IGNORECASE)
    testo = re.sub(r'Cordiali saluti.*',    '', testo, flags=re.DOTALL | re.IGNORECASE)
    testo = re.sub(r'\s+', ' ', testo).strip()
    parole = testo.split()
    if len(parole) > max_parole:
        testo = ' '.join(parole[:max_parole]) + '...'
    return testo


def classifica_area_fewshot(oggetto: str, descrizione: str) -> ClassificazioneArea:
    """Classifica un ticket usando GPT-4o-mini con few-shot prompt."""
    testo = preprocess_testo(oggetto, descrizione)

    response = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        temperature=0,
        max_tokens=120,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': f'OGGETTO: {oggetto}\nDESCRIZIONE: {testo}'}
        ],
        response_format=ClassificazioneArea,
    )
    return response.choices[0].message.parsed


print('Classificatore few-shot pronto.')
print('Test rapido su un esempio:')
test = classifica_area_fewshot(
    'Errore rapportino cooperativa L.381',
    'Il modulo cooperative non calcola correttamente le timbrate del mese di gennaio per i soci'
)
print(f'  Area:       {test.area}')
print(f'  Confidenza: {test.confidenza}')
print(f'  Reasoning:  {test.reasoning}')

Classificatore few-shot pronto.
Test rapido su un esempio:
  Area:       sistema381
  Confidenza: 0.9
  Reasoning:  La presenza della keyword 'L.381' indica che il ticket riguarda un errore nel modulo per cooperative sociali.


## STEP 9 — Valutazione few-shot sul test set

In [12]:
from sklearn.metrics import classification_report

N_CAMPIONI  = 296   # stesso campione del test zero-shot per confronto diretto
RANDOM_SEED = 42

# Filtra il test set sulle classi valide
df_test_valido = df_test[df_test['area_v2'].isin(CLASSI)].copy()
campioni       = df_test_valido.sample(n=N_CAMPIONI, random_state=RANDOM_SEED)

print(f'Campione: {N_CAMPIONI} ticket dal test set')
print(f'Distribuzione:\n{campioni["area_v2"].value_counts().to_string()}')

risultati = []
errori    = 0

for i, (idx, row) in enumerate(campioni.iterrows()):
    try:
        oggetto    = str(row.get('oggetto', ''))
        descrizione = str(row.get('descrizione_pulita', row.get('testo_input', '')))

        pred = classifica_area_fewshot(oggetto, descrizione)

        risultati.append({
            'idx':          idx,
            'area_reale':   row['area_v2'],
            'area_predetta': pred.area,
            'confidenza':   pred.confidenza,
            'reasoning':    pred.reasoning,
            'corretto':     row['area_v2'] == pred.area,
        })

        if (i + 1) % 50 == 0:
            acc = sum(r['corretto'] for r in risultati) / len(risultati)
            print(f'[{i+1}/{N_CAMPIONI}] Accuracy: {acc:.1%} | Errori API: {errori}')

        # Rate limiting cautela
        if (i + 1) % 100 == 0:
            time.sleep(0.5)

    except Exception as e:
        errori += 1
        print(f'  Errore su idx {idx}: {e}')

df_ris = pd.DataFrame(risultati)

print('\n' + '='*60)
print('RISULTATI GPT-4o-mini (FEW-SHOT)')
print('='*60)
print(classification_report(
    df_ris['area_reale'],
    df_ris['area_predetta'],
    zero_division=0
))
print(f'Accuracy: {df_ris["corretto"].mean():.4f}')
print(f'Errori API: {errori}/{N_CAMPIONI}')

df_ris.to_csv(OUTPUT_DIR / 'gpt_fewshot_risultati.csv', index=False, encoding='utf-8-sig')
print('Risultati salvati in: data/gpt_fewshot_risultati.csv')

Campione: 296 ticket dal test set
Distribuzione:
area_v2
area_personale                     98
ciclo_passivo                      66
ciclo_attivo                       51
area_sanitaria                     37
protocollo_documentale_delibere    15
rendicontazione_flussi             13
area_sistemistica                  10
sistema381                          3
area_territoriale                   3
[50/296] Accuracy: 78.0% | Errori API: 0
[100/296] Accuracy: 76.0% | Errori API: 0
[150/296] Accuracy: 72.7% | Errori API: 0
[200/296] Accuracy: 69.5% | Errori API: 0
[250/296] Accuracy: 69.6% | Errori API: 0

RISULTATI GPT-4o-mini (FEW-SHOT)
                                 precision    recall  f1-score   support

                 area_personale       0.95      0.84      0.89        98
                 area_sanitaria       0.66      0.78      0.72        37
              area_sistemistica       0.18      0.90      0.31        10
              area_territoriale       1.00      0.33      0.50   

## STEP 10 — Confronto diretto zero-shot vs few-shot vs LinearSVC

In [14]:
from sklearn.metrics import f1_score

# Carica risultati zero-shot precedenti
df_zs = pd.read_csv(OUTPUT_DIR / 'gpt_risultati.csv')
df_fs = pd.read_csv(OUTPUT_DIR / 'gpt_fewshot_risultati.csv')

# F1 per classe — zero-shot
f1_zs = f1_score(df_zs['area_reale'], df_zs['area_predetta'],
                  labels=CLASSI, average=None, zero_division=0)

# F1 per classe — few-shot
f1_fs = f1_score(df_fs['area_reale'], df_fs['area_predetta'],
                  labels=CLASSI, average=None, zero_division=0)

# LinearSVC dal metadata
with open(MODEL_DIR / 'metadata.json') as f:
    meta = json.load(f)

confronto = pd.DataFrame({
    'classe':     CLASSI,
    'zero_shot':  f1_zs.round(2),
    'few_shot':   f1_fs.round(2),
    'delta':      (f1_fs - f1_zs).round(2),
})

confronto['migliora'] = confronto['delta'] > 0
confronto = confronto.sort_values('delta', ascending=False)

print('\n' + '='*60)
print('CONFRONTO zero-shot vs few-shot')
print('='*60)
pd.set_option('display.float_format', '{:.2f}'.format)
print(confronto.to_string(index=False))

macro_zs = f1_score(df_zs['area_reale'], df_zs['area_predetta'], average='macro', zero_division=0)
macro_fs = f1_score(df_fs['area_reale'], df_fs['area_predetta'], average='macro', zero_division=0)

print(f'\nMacro F1 zero-shot: {macro_zs:.4f}')
print(f'Macro F1 few-shot:  {macro_fs:.4f}')
print(f'Delta Macro F1:     {macro_fs - macro_zs:+.4f}')
print(f'LinearSVC v2:        {meta["macro_f1_test"]:.4f}  (benchmark)')


CONFRONTO zero-shot vs few-shot
                         classe  zero_shot  few_shot  delta  migliora
              area_territoriale       0.25      0.50   0.25      True
         rendicontazione_flussi       0.55      0.76   0.21      True
protocollo_documentale_delibere       0.62      0.69   0.07      True
                  ciclo_passivo       0.67      0.69   0.03      True
                 area_personale       0.87      0.89   0.02      True
                 area_sanitaria       0.71      0.72   0.00     False
              area_sistemistica       0.31      0.31  -0.01     False
                   ciclo_attivo       0.71      0.56  -0.15     False
                     sistema381       0.67      0.50  -0.17     False

Macro F1 zero-shot: 0.5944
Macro F1 few-shot:  0.6240
Delta Macro F1:     +0.0296
LinearSVC v2:        0.6647  (benchmark)


In [15]:
# Diagnostico: dove finiscono i ticket ciclo_attivo classificati male?
df_err_att = df_ris[
    (df_ris['area_reale'] == 'ciclo_attivo') &
    (df_ris['area_predetta'] != 'ciclo_attivo')
]

print(f"Ticket ciclo_attivo sbagliati: {len(df_err_att)} / {(df_ris['area_reale']=='ciclo_attivo').sum()}")
print("\nDove li manda GPT:")
print(df_err_att['area_predetta'].value_counts())

print("\n--- Esempi degli errori (oggetto + reasoning GPT) ---")
for _, row in df_err_att.head(8).iterrows():
    idx = row['idx']
    testo = df_test.loc[idx, 'testo_input'][:120] if idx in df_test.index else '?'
    print(f"\n  Predetto: {row['area_predetta']}  |  Conf: {row['confidenza']:.2f}")
    print(f"  Reasoning: {row['reasoning']}")
    print(f"  Testo: {testo}")

Ticket ciclo_attivo sbagliati: 30 / 51

Dove li manda GPT:
area_predetta
ciclo_passivo                      11
area_sanitaria                      8
area_sistemistica                   7
rendicontazione_flussi              3
protocollo_documentale_delibere     1
Name: count, dtype: int64

--- Esempi degli errori (oggetto + reasoning GPT) ---

  Predetto: ciclo_passivo  |  Conf: 0.80
  Reasoning: Il ticket riguarda la generazione di file SEPA, che è legata alla gestione di pagamenti verso terzi.
  Testo: cartella di destinazione generazione sepa Buon pomeriggio.
Una volta generato i file sepa non li trovo più nella cartell

  Predetto: ciclo_passivo  |  Conf: 0.90
  Reasoning: Il ticket riguarda fatture scomparse dalla contabilità utenti, indicando un problema con la registrazione delle fatture passive.
  Testo: Fatture cancellate da contabilità utenti Buongiorno,
abbiamo un problema con due fatture che sono scomparse dalla contab

  Predetto: area_sistemistica  |  Conf: 0.80
  Reasonin